In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import json
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import torch.nn.functional as F
from tqdm import tqdm
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import re

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
# model_5th = Word2Vec.load("Word2Vec/Exp3/model_5th.model")
# model_12th = Word2Vec.load("Word2Vec/Exp3/model_12th.model")
model_13th = Word2Vec.load("Exp3/model_13th.model")
model_14th = Word2Vec.load("Exp3/model_14th.model")
model_15th = Word2Vec.load("Exp3/model_15th.model")
# model_16th = Word2Vec.load("Word2Vec/Exp3/model_16th.model")
model_18th = Word2Vec.load("Exp3/model_18th.model")
model_19th = Word2Vec.load("Exp3/model_19th.model")
model_20th = Word2Vec.load("Exp3/model_20th.model")

In [4]:
with open("../Data/consistent_keys_2.json", "r", encoding="utf-8") as f:
    consistent_keys = json.load(f)

In [5]:
len(consistent_keys)

210

In [6]:
century_models = {
    # '5th': model_5th,
    # '12th': model_12th,
    '13th': model_13th,
    '14th': model_14th,
    '15th': model_15th,
    # '16th': model_16th,
    '18th': model_18th,
    '19th': model_19th,
    '20th': model_20th
    }

In [7]:
century_names = sorted(list(century_models.keys()))

In [8]:
learned_weights_matrices = {}

In [9]:
# Hyperparameters for training
embedding_size = 50 
learning_rate = 0.001
epochs = 2000 
lambda_ortho = 0.001 # Penalty to keep W rotational (orthogonal)

In [10]:
# --- Configuration ---
anchor_name = "13th" 
patience_limit = 20        
patience_threshold = 1e-7  
min_epochs = 100           
report_interval = 50       # How often to print epoch updates

alignment_quality_metrics = []
unified_space = {}
learned_weights_matrices = {}

# 1. Setup the Anchor (13th Century)
raw_Y = torch.tensor([century_models[anchor_name].wv[word] for word in consistent_keys], dtype=torch.float32)
Y_ref = F.normalize(raw_Y, p=2, dim=1)
Y_ref_np = Y_ref.detach().numpy() 

unified_space[anchor_name] = Y_ref_np

for target_name in century_names:
    if target_name == anchor_name:
        continue
            
    # 2. Prepare Source Vectors (Target Century)
    raw_X = torch.tensor([century_models[target_name].wv[word] for word in consistent_keys], dtype=torch.float32)
    X_input = F.normalize(raw_X, p=2, dim=1)

    # 3. Model Setup: Map Target -> Anchor Space
    model = nn.Linear(embedding_size, embedding_size, bias=False)
    nn.init.eye_(model.weight)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    
    best_loss = float('inf')
    patience_counter = 0
    
    print(f"\n" + "="*50)
    print(f"ALIGNING: {target_name} ---> {anchor_name}")
    print("="*50)

    # 4. Training Loop
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_input)
        
        mse_loss = criterion(predictions, Y_ref)
        
        # Orthogonality Constraint
        W_curr = model.weight
        ortho_loss = torch.norm(torch.mm(W_curr, W_curr.t()) - torch.eye(embedding_size), p='fro')**2
        
        loss = mse_loss + (lambda_ortho * ortho_loss)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()

        # Print Epoch Progress
        if (epoch + 1) % report_interval == 0 or epoch == 0:
            print(f"Epoch {epoch+1:4d} | Total Loss: {current_loss:.7f} | MSE: {mse_loss.item():.7f} | Ortho: {ortho_loss.item():.7f}")

        # --- Early Stopping Logic ---
        if (best_loss - current_loss) > patience_threshold:
            best_loss = current_loss
            patience_counter = 0
        else:
            patience_counter += 1

        # Check for Convergence after min_epochs
        if patience_counter >= patience_limit and epoch >= min_epochs:
            print(f"--- Converged at epoch {epoch+1} ---")
            break

    # 5. Final Alignment and Quality Measurement
    with torch.no_grad():
        Q = model.weight.detach().numpy()
        # Transform the target space into the anchor space
        X_aligned = (X_input @ torch.tensor(Q).t()).numpy()
        
        unified_space[target_name] = X_aligned
        learned_weights_matrices[f"{target_name}_to_{anchor_name}"] = Q

        # --- INFUSION: MEASURE ALIGNMENT QUALITY ---
        # Procrustes Disparity (Sum of Squared Errors)
        disparity = np.sum(np.square(X_aligned - Y_ref_np))
        
        # Rotational Distance (Frobenius norm of difference from Identity)
        rotational_distance = np.linalg.norm(Q - np.eye(embedding_size), ord='fro')

        alignment_quality_metrics.append({
            "Target_Century": target_name,
            "Reference_Century": anchor_name,
            "Procrustes_Disparity": round(float(disparity), 6),
            "Rotational_Distance": round(float(rotational_distance), 6),
            "Epochs_Completed": epoch + 1
        })

# Final Summary Table
metrics_df = pd.DataFrame(alignment_quality_metrics)
print("\n" + "!"*30)
print("FINAL ALIGNMENT REPORT")
print("!"*30)
print(metrics_df)

/var/folders/p0/y_n4880x7cjbhx11tzvvqpzh0000gn/T/ipykernel_37749/399615730.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  raw_Y = torch.tensor([century_models[anchor_name].wv[word] for word in consistent_keys], dtype=torch.float32)



ALIGNING: 14th ---> 13th
Epoch    1 | Total Loss: 0.0319770 | MSE: 0.0319770 | Ortho: 0.0000000
Epoch   50 | Total Loss: 0.0223909 | MSE: 0.0213016 | Ortho: 1.0892733
Epoch  100 | Total Loss: 0.0210490 | MSE: 0.0201291 | Ortho: 0.9198687
Epoch  150 | Total Loss: 0.0202421 | MSE: 0.0193944 | Ortho: 0.8477527
Epoch  200 | Total Loss: 0.0195878 | MSE: 0.0187851 | Ortho: 0.8027049
Epoch  250 | Total Loss: 0.0190390 | MSE: 0.0182676 | Ortho: 0.7714135
Epoch  300 | Total Loss: 0.0185739 | MSE: 0.0178257 | Ortho: 0.7482381
Epoch  350 | Total Loss: 0.0181763 | MSE: 0.0174462 | Ortho: 0.7301151
Epoch  400 | Total Loss: 0.0178335 | MSE: 0.0171180 | Ortho: 0.7155086
Epoch  450 | Total Loss: 0.0175357 | MSE: 0.0168320 | Ortho: 0.7036639
Epoch  500 | Total Loss: 0.0172753 | MSE: 0.0165812 | Ortho: 0.6940961
Epoch  550 | Total Loss: 0.0170463 | MSE: 0.0163599 | Ortho: 0.6863769
Epoch  600 | Total Loss: 0.0168439 | MSE: 0.0161638 | Ortho: 0.6801205
Epoch  650 | Total Loss: 0.0166645 | MSE: 0.0159895

In [11]:
print(f"Learned {len(learned_weights_matrices)} transformation matrices (W).")

Learned 5 transformation matrices (W).


In [12]:
metrics_df

,Target_Century,Reference_Century,Procrustes_Disparity,Rotational_Distance,Epochs_Completed
0,14th,13th,153.117615,9.137767,2000
1,15th,13th,159.407486,9.301473,2000
2,18th,13th,140.859116,8.770955,2000
3,19th,13th,165.559387,9.215001,2000
4,20th,13th,166.860413,9.352003,2000


In [13]:
avg_procrustes = metrics_df['Procrustes_Disparity'].mean()
avg_rotational = metrics_df['Rotational_Distance'].mean()

print(f"Average Procrustes Disparity: {avg_procrustes:.4f}")
print(f"Average Rotational Distance: {avg_rotational:.4f}")

Average Procrustes Disparity: 157.1608
Average Rotational Distance: 9.1554


In [30]:
metrics_df.to_latex('WV_NOP.md', index=False)

In [15]:
output_dir = '../Data/Word_Similarity_Heatmaps_Method_3_WV'

In [16]:
font_path = '../Fonts/Noto_Sans_Sinhala/NotoSansSinhala-VariableFont_wdth,wght.ttf'
fm.fontManager.addfont(font_path)
sinhala_font_name = fm.FontProperties(fname=font_path).get_name()
fm._load_fontmanager(try_read_cache=False)

Matplotlib is building the font cache; this may take a moment.


In [17]:
# 1. Map consistent_keys to their index for quick lookup
word_to_idx = {word: i for i, word in enumerate(consistent_keys)}
centuries = sorted(unified_space.keys()) # Ensure chronological order

for word in tqdm(consistent_keys, desc="Generating Heatmaps"):
    word_idx = word_to_idx[word]
    
    # 2. Build the Similarity Matrix for this specific word
    # Rows = Year A, Cols = Year B
    sim_matrix = np.zeros((len(centuries), len(centuries)))
    
    for i, year_a in enumerate(centuries):
        for j, year_b in enumerate(centuries):
            # Extract word vectors from the unified space
            vec_a = unified_space[year_a][word_idx].reshape(1, -1)
            vec_b = unified_space[year_b][word_idx].reshape(1, -1)
            
            # Calculate Similarity
            sim = cosine_similarity(vec_a, vec_b)[0][0]
            sim_matrix[i, j] = sim
            
    # 3. Convert to DataFrame for Seaborn
    heatmap_df = pd.DataFrame(sim_matrix, index=centuries, columns=centuries)

    # 4. Plotting
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        heatmap_df, 
        annot=True,              
        fmt=".3f",               
        cmap="YlGnBu",           
        linewidths=0.5,          
        linecolor='white',       
        cbar_kws={'label': 'Unified Space Cosine Similarity'} 
    )

    plt.title(f'Cross-Century Semantic Stability: "{word}"', fontsize=16, fontproperties=sinhala_font_name)
    plt.xlabel('Century B', fontsize=12, fontproperties=sinhala_font_name)
    plt.ylabel('Century A', fontsize=12, fontproperties=sinhala_font_name)

    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # 5. Save logic
    safe_key = str(word).replace(' ', '_').replace('/', '_')
    filename = os.path.join(output_dir, f"{safe_key}_similarity.png")
    
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

print(f"\nSuccess: {len(consistent_keys)} heatmaps saved to '{output_dir}'.")

Generating Heatmaps: 100%|██████████| 210/210 [00:50<00:00,  4.12it/s]


Success: 210 heatmaps saved to '../Data/Word_Similarity_Heatmaps_Method_3_WV'.


In [18]:
centuries = sorted(unified_space.keys(), key=lambda x: int(''.join(filter(str.isdigit, x))))
word_to_idx = {word: i for i, word in enumerate(consistent_keys)}

all_rows = []

# 2. Iterate through each word
for word in consistent_keys:
    word_idx = word_to_idx[word]
    
    # 3. For each word, create a row for every 'Year A'
    for year_a in centuries:
        row_data = {
            'Key': word,
            'Year_A': year_a
        }
        
        # Get vector for Year A
        vec_a = unified_space[year_a][word_idx].reshape(1, -1)
        
        # 4. Compare Year A against all possible 'Year B's
        for year_b in centuries:
            vec_b = unified_space[year_b][word_idx].reshape(1, -1)
            sim = cosine_similarity(vec_a, vec_b)[0][0]
            
            # Use the century name as the column name (Year B)
            row_data[year_b] = round(float(sim), 6)
            
        all_rows.append(row_data)

# 5. Create the final DataFrame
final_df_org = pd.DataFrame(all_rows)

# 6. Sort by Key (Word) and then chronologically by Year_A
final_df_org = final_df_org.sort_values(by=['Key', 'Year_A']).reset_index(drop=True)

In [19]:
final_df_org.head(10)

,Key,Year_A,13th,14th,15th,18th,19th,20th
0,අක්ෂර,13th,1.000000,0.750987,0.637515,0.671267,0.647581,0.599460
1,අක්ෂර,14th,0.750987,1.000000,0.737562,0.728435,0.632548,0.658833
2,අක්ෂර,15th,0.637515,0.737562,1.000000,0.943555,0.708384,0.651642
3,අක්ෂර,18th,0.671267,0.728435,0.943555,1.000000,0.680974,0.616187
4,අක්ෂර,19th,0.647581,0.632548,0.708384,0.680974,1.000000,0.547944
5,අක්ෂර,20th,0.599460,0.658833,0.651642,0.616187,0.547944,1.000000
6,අග්ර,13th,1.000000,0.343397,0.597844,0.547788,0.569667,0.624136
7,අග්ර,14th,0.343397,1.000000,0.461095,0.517200,0.470393,0.513088
8,අග්ර,15th,0.597844,0.461095,1.000000,0.527868,0.490492,0.678245
9,අග්ර,18th,0.547788,0.517200,0.527868,1.000000,0.742905,0.578729


In [20]:
def extract_year(name):
    try:
        return int(''.join(filter(str.isdigit, str(name))))
    except ValueError:
        return 0

In [21]:
df_long = final_df_org.melt(id_vars=['Key', 'Year_A'], var_name='Year_B', value_name='Value')
df_long['Year_A_num'] = df_long['Year_A'].apply(extract_year)
df_long['Year_B_num'] = df_long['Year_B'].apply(extract_year)

df_long['feature_pair'] = df_long['Year_A'].astype(str) + '_' + df_long['Year_B'].astype(str)

df_lags_only = df_long[df_long['Year_A_num'] < df_long['Year_B_num']].copy()

piv_df_org = df_lags_only.pivot(index='Key', columns='feature_pair', values='Value')
piv_df_org['Mean_Similarity'] = piv_df_org.mean(axis=1, numeric_only=True)
piv_df_org = piv_df_org.reset_index()

In [22]:
piv_df_org.head()

feature_pair,Key,13th_14th,13th_15th,13th_18th,13th_19th,13th_20th,14th_15th,14th_18th,14th_19th,14th_20th,15th_18th,15th_19th,15th_20th,18th_19th,18th_20th,19th_20th,Mean_Similarity
0,අක්ෂර,0.750987,0.637515,0.671267,0.647581,0.599460,0.737562,0.728435,0.632548,0.658833,0.943555,0.708384,0.651642,0.680974,0.616187,0.547944,0.680858
1,අග්ර,0.343397,0.597844,0.547788,0.569667,0.624136,0.461095,0.517200,0.470393,0.513088,0.527868,0.490492,0.678245,0.742905,0.578729,0.597064,0.550661
2,අතුරෙන්,0.569270,0.508499,0.532126,0.502062,0.767390,0.484718,0.722913,0.422614,0.497948,0.557707,0.442052,0.383214,0.603134,0.526692,0.420478,0.529388
3,අන්ය,0.582882,0.610777,0.597556,0.633016,0.528410,0.510752,0.500618,0.473763,0.370202,0.360130,0.463164,0.424968,0.478557,0.488434,0.473497,0.499782
4,අනෙක,0.714044,0.583493,0.720664,0.587922,0.651551,0.524156,0.631641,0.459905,0.622955,0.569418,0.548096,0.425177,0.655222,0.766258,0.661036,0.608103


In [23]:
piv_df_org['Mean_Similarity'].describe()

count    210.000000
mean       0.529154
std        0.047610
min        0.395756
25%        0.498260
50%        0.529523
75%        0.562222
max        0.680858
Name: Mean_Similarity, dtype: float64

In [24]:
def get_semantic_drift_report(df, models_dict, threshold_std=0.8, top_n=10):
    results = []
    # Ensure 'Key' is the index
    work_df = df.set_index('Key') if 'Key' in df.columns else df.copy()
    
    # Identify all comparison columns (13_14, 13_15, 14_15, etc.)
    cols = [c for c in work_df.columns if isinstance(c, str) and '_' in c]
    
    for col in cols:
        parts = col.split('_')
        try:
            y_start, y_end = int(parts[0]), int(parts[1])
        except: continue
        
        if y_start not in models_dict or y_end not in models_dict:
            continue

        m_start = models_dict[y_start]
        m_end = models_dict[y_end]

        # Calculate a dynamic cutoff for this specific period
        series = work_df[col][work_df[col] > 0]
        if series.empty: continue
        
        cutoff = series.mean() - (threshold_std * series.std())
        dips = series[series < cutoff]
        
        for word, score in dips.items():
            try:
                # 1. Check vocab existence
                if word not in m_start.wv or word not in m_end.wv:
                    continue
                
                # 2. Dynamic top_n check to prevent IndexError
                # If vocab is smaller than requested top_n, adjust it
                v_start_size = len(m_start.wv.index_to_key)
                v_end_size = len(m_end.wv.index_to_key)
                safe_n = min(top_n, v_start_size - 1, v_end_size - 1)
                
                if safe_n < 1: continue

                # 3. Retrieve neighbors with explicit error handling for the Gensim bug
                try:
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                except IndexError:
                    # If Gensim's index is still broken, force a refresh and try one last time
                    m_start.wv.fill_norms(force=True)
                    m_end.wv.fill_norms(force=True)
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                
                n_start = [n[0] for n in n_start_raw]
                n_end = [n[0] for n in n_end_raw]
                
                results.append({
                    "Word": word,
                    "Period": col,
                    "Similarity": round(score, 4),
                    f"Neighbors_{y_start}": ", ".join(n_start),
                    f"Neighbors_{y_end}": ", ".join(n_end)
                })
            except Exception:
                # Catch-all for any remaining weirdness with specific words
                continue

    return pd.DataFrame(results)

In [25]:
# --- 1. Create a temporary models dictionary with integer keys ---
# This converts {"13th_century": model} -> {13: model}
int_models_dict = {}
for key, model in century_models.items():
    # Extract just the digits (e.g., "13th_century" -> 13)
    int_key = int(''.join(filter(str.isdigit, key)))
    int_models_dict[int_key] = model

# --- 2. Create a temporary DataFrame with integer column names ---
# This converts "13th_century_14th_century" -> "13_14"
piv_df_translated = piv_df_org.copy()
new_cols = {'Key': 'Key'}

for col in piv_df_org.columns:
    if col == 'Key': continue
    # Find all numbers in the column string
    nums = re.findall(r'\d+', col)
    if len(nums) >= 2:
        new_cols[col] = f"{nums[0]}_{nums[1]}"

piv_df_translated = piv_df_translated.rename(columns=new_cols)

# --- 3. Run your exact function without changes ---
semantic_report = get_semantic_drift_report(
    piv_df_translated, 
    int_models_dict, 
    threshold_std=0.8
)

# --- 4. Final Polish ---
if not semantic_report.empty:
    semantic_report = semantic_report.sort_values(by=['Period', 'Similarity'])
    print(f"Success! Detected {len(semantic_report)} drifting words.")
    display(semantic_report.head(10))
else:
    print("No drifts detected. Try lowering threshold_std to 0.5.")

Success! Detected 648 drifting words.


,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,අග්ර,13_14,0.3434,"පාර්ෂණිය, තලය, ඔබන, කලැ, පාර්ශවය, උපස්ථායක, ජි...","බොධිවජ්රාශන, මව්පිය, සාගරයෙන්, දායක, උපදනා, ගො...",NaN,NaN,NaN,NaN
5,ඒ,13_14,0.3441,"ප්රශ්නයාගේ, ඉ, සංක්ෂෙපාර්ථයි, මුලුල්ලම, අවසානය...","රහතුන්, දේශනා, තිස්දෙනා, මහණෙනි, එම, දෙනෙකුට, ...",NaN,NaN,NaN,NaN
35,හට,13_14,0.3662,"විදර්ශකය, ශමථයානිකය, ශ්රවණාදී, අධ්යයන, ශිෂ්ය, ...","ගෙවා, පන්තිස්වන, පිළිගෙන, උයන්හි, සත්සති, බ්රහ...",NaN,NaN,NaN,NaN
21,පැමිණි,13_14,0.3759,"කොටසෙහි, පිළිගෙන, දිවජස්, දෙවිව, දෙවියන්, බහාල...","එපවත්, පනවන, මහණෙක්, වහස්නා, ගණයා, කියනුයේ, පර...",NaN,NaN,NaN,NaN
33,සහිත,13_14,0.3873,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ, මේලෙසින්, ...","වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්...",NaN,NaN,NaN,NaN
10,කියා,13_14,0.3913,"කිමවන, පළමුවෙනි, සරෙහි, වදෙය්ය, නුසස්, මරණෙහි,...","අර්හත්, නිශ්රය, සමෘද්ධිය, ප්රාසාදය, කළකිරී, කැ...",NaN,NaN,NaN,NaN
3,උපන්,13_14,0.4036,"නාගව, ලොව, නාග, භවනයෙන්, අවුලෙන්, උයන, වතු, ඇත...","දූතයන්, ව්යාධි, පුත්රුවනක්, මත, කිරි, ව්රසහක්ඛ...",NaN,NaN,NaN,NaN
8,කල්ප,13_14,0.4057,"සොයා, සම්බුදුරජානන්, බෝමැඩ, හිඳු, මාත්සර්යය, ම...","නොවර, තුසී, නිරතුරුව, එකාකාර, පාරමීධර්මයන්, දව...",NaN,NaN,NaN,NaN
24,මහා,13_14,0.4204,"ව්යාඛ්යානය, යුගලයෙන්, ජය, අනිමිසලෝචන, තත, පුදන...","චරිතයක්, ගෙනහැර, ශාසනාවතාර, ස්ථවිරපාදයෝ, දක්වත...",NaN,NaN,NaN,NaN
4,එම,13_14,0.4230,"ආර්යය, භූමියෙහි, ගතයන්, අධිගතයන්, රූපයහ, දශක, ...","භාගයෙහි, රාත්රි, වර, මෙන්නාහු, ශ්රොතාපත්ති, ප්...",NaN,NaN,NaN,NaN


In [26]:
display(semantic_report.head(30))

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,අග්ර,13_14,0.3434,"පාර්ෂණිය, තලය, ඔබන, කලැ, පාර්ශවය, උපස්ථායක, ජි...","බොධිවජ්රාශන, මව්පිය, සාගරයෙන්, දායක, උපදනා, ගො...",NaN,NaN,NaN,NaN
5,ඒ,13_14,0.3441,"ප්රශ්නයාගේ, ඉ, සංක්ෂෙපාර්ථයි, මුලුල්ලම, අවසානය...","රහතුන්, දේශනා, තිස්දෙනා, මහණෙනි, එම, දෙනෙකුට, ...",NaN,NaN,NaN,NaN
35,හට,13_14,0.3662,"විදර්ශකය, ශමථයානිකය, ශ්රවණාදී, අධ්යයන, ශිෂ්ය, ...","ගෙවා, පන්තිස්වන, පිළිගෙන, උයන්හි, සත්සති, බ්රහ...",NaN,NaN,NaN,NaN
21,පැමිණි,13_14,0.3759,"කොටසෙහි, පිළිගෙන, දිවජස්, දෙවිව, දෙවියන්, බහාල...","එපවත්, පනවන, මහණෙක්, වහස්නා, ගණයා, කියනුයේ, පර...",NaN,NaN,NaN,NaN
33,සහිත,13_14,0.3873,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ, මේලෙසින්, ...","වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්...",NaN,NaN,NaN,NaN
10,කියා,13_14,0.3913,"කිමවන, පළමුවෙනි, සරෙහි, වදෙය්ය, නුසස්, මරණෙහි,...","අර්හත්, නිශ්රය, සමෘද්ධිය, ප්රාසාදය, කළකිරී, කැ...",NaN,NaN,NaN,NaN
3,උපන්,13_14,0.4036,"නාගව, ලොව, නාග, භවනයෙන්, අවුලෙන්, උයන, වතු, ඇත...","දූතයන්, ව්යාධි, පුත්රුවනක්, මත, කිරි, ව්රසහක්ඛ...",NaN,NaN,NaN,NaN
8,කල්ප,13_14,0.4057,"සොයා, සම්බුදුරජානන්, බෝමැඩ, හිඳු, මාත්සර්යය, ම...","නොවර, තුසී, නිරතුරුව, එකාකාර, පාරමීධර්මයන්, දව...",NaN,NaN,NaN,NaN
24,මහා,13_14,0.4204,"ව්යාඛ්යානය, යුගලයෙන්, ජය, අනිමිසලෝචන, තත, පුදන...","චරිතයක්, ගෙනහැර, ශාසනාවතාර, ස්ථවිරපාදයෝ, දක්වත...",NaN,NaN,NaN,NaN
4,එම,13_14,0.4230,"ආර්යය, භූමියෙහි, ගතයන්, අධිගතයන්, රූපයහ, දශක, ...","භාගයෙහි, රාත්රි, වර, මෙන්නාහු, ශ්රොතාපත්ති, ප්...",NaN,NaN,NaN,NaN


In [27]:
semantic_report.describe()

,Similarity
count,648.000000
mean,0.391512
std,0.081197
min,0.129100
25%,0.336000
50%,0.387300
75%,0.461275
max,0.523000


In [28]:
semantic_report.to_csv('../Data/word2vec_dip_neighbors_orthogonal_procrustes_NN.csv', index=False, encoding='utf-8-sig')

In [31]:
import pandas as pd
import numpy as np

In [32]:
semantic_report = pd.read_csv('../Data/word2vec_dip_neighbors_orthogonal_procrustes_NN.csv', encoding='utf-8-sig')

In [33]:
semantic_report.head()

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,අග්ර,13_14,0.3434,"පාර්ෂණිය, තලය, ඔබන, කලැ, පාර්ශවය, උපස්ථායක, ජි...","බොධිවජ්රාශන, මව්පිය, සාගරයෙන්, දායක, උපදනා, ගො...",NaN,NaN,NaN,NaN
1,ඒ,13_14,0.3441,"ප්රශ්නයාගේ, ඉ, සංක්ෂෙපාර්ථයි, මුලුල්ලම, අවසානය...","රහතුන්, දේශනා, තිස්දෙනා, මහණෙනි, එම, දෙනෙකුට, ...",NaN,NaN,NaN,NaN
2,හට,13_14,0.3662,"විදර්ශකය, ශමථයානිකය, ශ්රවණාදී, අධ්යයන, ශිෂ්ය, ...","ගෙවා, පන්තිස්වන, පිළිගෙන, උයන්හි, සත්සති, බ්රහ...",NaN,NaN,NaN,NaN
3,පැමිණි,13_14,0.3759,"කොටසෙහි, පිළිගෙන, දිවජස්, දෙවිව, දෙවියන්, බහාල...","එපවත්, පනවන, මහණෙක්, වහස්නා, ගණයා, කියනුයේ, පර...",NaN,NaN,NaN,NaN
4,සහිත,13_14,0.3873,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ, මේලෙසින්, ...","වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්...",NaN,NaN,NaN,NaN


In [34]:
sorted_report = semantic_report.sort_values(by='Similarity', ascending=True)

In [35]:
sorted_report.head(6)

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
212,පැමිණි,14_15,0.1291,NaN,"එපවත්, පනවන, මහණෙක්, වහස්නා, ගණයා, කියනුයේ, පර...","සිව්මැලි, ඔප, වැඩි, සඳමෙන්, හෙල්, මැලි, පින්කළ...",NaN,NaN,NaN
294,රාජ,14_19,0.1501,NaN,"විභූෂිත, මණිමකුට, ස්වස්ති, රාජවංශ, පීඨ, රඤ්ජිත...",NaN,NaN,"නගුට, තැනැත්තේ, කුමාරිකා, දුරදීම, කන්, ආලය, සල...",NaN
425,දෙන,15_19,0.1537,NaN,NaN,"ප්රදානය, සම්පත්ති, ගාත්රයක්, කාන්තියෙන්, අතිරි...",NaN,"වහන්සේනමකට, සැලකියයුතු, අවසර, ගිහියෙකුට, පැමිණ...",NaN
426,හා,15_19,0.1564,NaN,NaN,"වළඳනා, එකාංශ, ශිරො, ධාතුව, පාංශුකූල, ඌර්ධව, සු...",NaN,"වැලි, සයක්, පණස්සයකින්, පසක්, පණස්හයකින්, එකකි...",NaN
604,කෙනෙක්,19_20,0.1567,NaN,NaN,NaN,NaN,"ඇත්තාය, හැඟීයන්ට, මැවුම්කාරයා, සියල්ලන්ටම, දුර...","ප්රණක්ෂර, ලිප, ප්රීතියෙන්, නොහික්මුණු, කඩදාසිය..."
472,සිට,15_20,0.1751,NaN,NaN,"ගත්තා, දුවන්නා, වර්ණව, නැගී, ප්රභා, මණ්ඩලය, භි...",NaN,NaN,"අටවමින්, මරුගුල්, වයසේ, අයකු, ගොඩේ, සීමාවේ, සු..."


In [36]:
semantic_report['Similarity'] = semantic_report['Similarity'].round(4)
split_periods = semantic_report['Period'].str.split('_', expand=True)
ref_num = split_periods[0]
target_num = split_periods[1]

semantic_report['Reference Century'] = ref_num + 'th'
semantic_report['Target Century'] = target_num + 'th'

In [37]:
def get_top_5_from_col(row, century_num):
    col_name = f'Neighbors_{century_num}'
    
    # Check if the column exists and has data for this row
    if col_name in row.index and pd.notna(row[col_name]) and row[col_name] != '':
        # Split by comma, clean up spaces, and grab the first 5 words
        words = [w.strip() for w in str(row[col_name]).split(',') if w.strip()]
        return ', '.join(words[:5])
    
    return '' # Return empty if no data found

# 4. Extract neighbors dynamically for both Reference and Target
semantic_report['Ref Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[0]), axis=1
)

semantic_report['Target Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[1]), axis=1
)


In [38]:
final_df = semantic_report[[
    'Word', 
    'Similarity',
    'Reference Century', 
    'Ref Century Neighbors', 
    'Target Century', 
    'Target Century Neighbors'
]].copy()

In [39]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
0,අග්ර,0.3434,13th,"පාර්ෂණිය, තලය, ඔබන, කලැ, පාර්ශවය",14th,"බොධිවජ්රාශන, මව්පිය, සාගරයෙන්, දායක, උපදනා"
1,ඒ,0.3441,13th,"ප්රශ්නයාගේ, ඉ, සංක්ෂෙපාර්ථයි, මුලුල්ලම, අවසානයෙහි",14th,"රහතුන්, දේශනා, තිස්දෙනා, මහණෙනි, එම"
2,හට,0.3662,13th,"විදර්ශකය, ශමථයානිකය, ශ්රවණාදී, අධ්යයන, ශිෂ්ය",14th,"ගෙවා, පන්තිස්වන, පිළිගෙන, උයන්හි, සත්සති"
3,පැමිණි,0.3759,13th,"කොටසෙහි, පිළිගෙන, දිවජස්, දෙවිව, දෙවියන්",14th,"එපවත්, පනවන, මහණෙක්, වහස්නා, ගණයා"
4,සහිත,0.3873,13th,"ඵලගූ, උපසපනුන්, ඉහි, ග්රහණ, ප්රකාශ",14th,"වාකචීරාදීන්, තිබූ, ක්ෂීණාශ්රවයන්, දිවිසම්, සන්තු"


In [40]:
final_df = final_df.sort_values(by='Similarity', ascending=True)

In [41]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
212,පැමිණි,0.1291,14th,"එපවත්, පනවන, මහණෙක්, වහස්නා, ගණයා",15th,"සිව්මැලි, ඔප, වැඩි, සඳමෙන්, හෙල්"
294,රාජ,0.1501,14th,"විභූෂිත, මණිමකුට, ස්වස්ති, රාජවංශ, පීඨ",19th,"නගුට, තැනැත්තේ, කුමාරිකා, දුරදීම, කන්"
425,දෙන,0.1537,15th,"ප්රදානය, සම්පත්ති, ගාත්රයක්, කාන්තියෙන්, අතිරික්ත",19th,"වහන්සේනමකට, සැලකියයුතු, අවසර, ගිහියෙකුට, පැමිණ..."
426,හා,0.1564,15th,"වළඳනා, එකාංශ, ශිරො, ධාතුව, පාංශුකූල",19th,"වැලි, සයක්, පණස්සයකින්, පසක්, පණස්හයකින්"
604,කෙනෙක්,0.1567,19th,"ඇත්තාය, හැඟීයන්ට, මැවුම්කාරයා, සියල්ලන්ටම, දුර්වල",20th,"ප්රණක්ෂර, ලිප, ප්රීතියෙන්, නොහික්මුණු, කඩදාසිය"


In [42]:
final_df['Ref Century Neighbors'] = '\\iffalse ' + final_df['Ref Century Neighbors'].astype(str) + ' \\fi'
final_df['Target Century Neighbors'] = '\\iffalse ' + final_df['Target Century Neighbors'].astype(str) + ' \\fi'

In [43]:
float_formatter = lambda x: f"{x:.4f}".rstrip('0').rstrip('.') if isinstance(x, float) else str(x)

In [45]:
final_df.head(5).to_latex('../Data/Word2Vec_Semantic_Drift_Report_NPA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})

In [46]:
final_df.to_csv('../Data/Word2Vec_dip_neighbors_similarity_matrix_NPA_final.csv', index=False, encoding='utf-8-sig')    